# Проект 1. Аналитическая записка

## Часть 1. Постановка бизнес-задачи

**ПРАЙМ** — платная подписка цифровой экосистемы, предоставляющая клиентам доступ к партнёрским сервисам: онлайн-кинотеатр, музыка, доставка продуктов и товаров, каршеринг, аптека, связь. Подпиской можно поделиться с близкими. За покупки в партнёрских сервисах начисляется кэшбэк.

Источники доходов: абонентская плата, комиссия с оборота в партнёрских сервисах, рост транзакционной активности подписчиков. Основные статьи расходов: привлечение клиентов, кэшбэк, выплаты партнёрам.

Два тарифа — базовый и расширенный. Есть триальный период.

### 1.1. Подключение

Для проведения исследования используется учебная база данных **PostgreSQL**, содержащая информацию о клиентах, подписках, платежах и других операциях сервиса ПРАЙМ.

Данные обрабатываются с помощью языка **Python** и SQL-запросов. Для работы с базой используются следующие библиотеки и инструменты:

- **pandas (`pd`)** — для хранения и анализа данных в формате DataFrame.
- **SQLAlchemy (`text`)** — для выполнения SQL-запросов к PostgreSQL.
- **prime.config (`get_engine`)** — функция, которая создаёт объект подключения к базе на основе настроек из файла `.env`.
- **sys** — для получения информации об используемом интерпретаторе Python при возникновении ошибки подключения.
- **matplotlib.pyplot, matplotlib.dates** — дополнительные библиотеки для построения графиков.

**Порядок подключения:**

1. Импортируются необходимые библиотеки и функция `get_engine()`.
2. С помощью `get_engine()` создаётся объект `engine`, содержащий настройки подключения к PostgreSQL.
3. Определяется вспомогательная функция `q(sql, **params)`, которая принимает SQL-запрос и возвращает результат в формате `pandas.DataFrame`.
4. При выполнении запроса открывается соединение с базой и устанавливается часовой пояс UTC. Это необходимо для единообразной работы с датами и временем.
5. SQL-запрос выполняется с помощью `pd.read_sql()`, а результат преобразуется в таблицу DataFrame.
6. После выполнения запроса соединение освобождается, чтобы не расходовать ресурсы общей учебной базы.

**Проверка подключения:** выполняется запрос `SELECT current_user AS login`, который возвращает имя пользователя PostgreSQL. Успешное получение логина подтверждает, что соединение с базой установлено.

Пароль и другие параметры доступа не записываются непосредственно в ноутбуке: они считываются из локального файла `.env`.

**Обработка ошибок:** если Python не находит пакет `prime`, код выводит подсказку о необходимости запуска Jupyter в окружении учебного репозитория.

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sqlalchemy import text

try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()

def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

### 1.2. Fast overview - обзор ключевых показателей бизнеса
На первом этапе исследования проводится **обзор ключевых показателей деятельности подписочного сервиса ПРАЙМ**: выручки, клиентской базы, подписок, удержания, каналов привлечения и использования сервисов.

Цель предварительного анализа — **выявить значимые изменения и закономерности**, которые могут указывать на потенциальные бизнес-проблемы или возможности для роста.

#### Общий масштаб бизнеса
Сначала посмотрим общую характеристику ПРАЙМ: сколько клиентов зарегистрировано, сколько подписок создано, сколько успешных платежей совершено и какова суммарная выручка.

In [ ]:
overview_total = q("""
    SELECT
        (SELECT COUNT(*) FROM prime.clients)
            AS total_clients,

        (SELECT COUNT(*) FROM prime.subscriptions)
            AS total_subscriptions,

        (SELECT COUNT(*)
         FROM prime.subscriptions
         WHERE ended_at IS NULL)
            AS subscriptions_without_end,

        (SELECT COUNT(*)
         FROM prime.payments
         WHERE status = 'success')
            AS successful_payments,

        (SELECT SUM(amount)
         FROM prime.payments
         WHERE status = 'success')
            AS total_revenue
""")

display(
    overview_total.style.format(
        lambda x: f"{x:,.0f}".replace(",", " ")
    )
)

_В базе данных зарегистрировано около **500 тыс. клиентов**, создано **333,9 тыс. подписок**, из которых приблизительно **168 тыс. не имеют зафиксированной даты завершения**. За доступный период зарегистрировано около **3 млн успешных платежей** на общую сумму **880,7 млн руб.**_

_Эти показатели характеризуют накопленный масштаб бизнеса, но не отражают число уникальных платящих клиентов или активных подписок на конкретную дату. Для оценки динамики бизнеса далее рассмотрим месячные показатели выручки, регистраций, подписок и использования сервисов._

#### Выручка и успешные платежи
Посмотрим динамику изменения выручки, кол-ва успешных платежей и среднего платежа.

In [ ]:
monthly_payments = q("""
    SELECT
        date_trunc('month', paid_at)::date AS month,
        COUNT(*) AS payment_count,
        SUM(amount) AS revenue,
        AVG(amount) AS avg_payment
    FROM prime.payments
    WHERE status = 'success'
    GROUP BY date_trunc('month', paid_at)::date
    ORDER BY month
""")

monthly_payments["month"] = pd.to_datetime(
    monthly_payments["month"]
)

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# 1. Создаём график
fig, ax1 = plt.subplots(figsize=(16, 7))

# Количество платежей — голубые столбцы
bars = ax1.bar(
    monthly_payments["month"],
    monthly_payments["payment_count"],
    width=20,
    color="lightsteelblue",
    alpha=0.7
)

ax1.set_ylabel("Количество успешных платежей, шт.")
ax1.set_ylim(0, monthly_payments["payment_count"].max() * 1.25)

# Количество платежей — подписи НАД столбцами
for bar in bars:
    height = bar.get_height()

    ax1.annotate(
        f"{height / 1000:.1f}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8,
        color="steelblue",
        fontweight="bold"
    )

# 2. Выручка — тёмно-синяя линия
ax2 = ax1.twinx()

revenue_mln = monthly_payments["revenue"] / 1_000_000

ax2.plot(
    monthly_payments["month"],
    revenue_mln,
    color="darkblue",
    marker="o",
    linewidth=2
)

ax2.set_ylabel("Выручка, млн ₽")
ax2.set_ylim(0, revenue_mln.max() * 1.25)

# 3. Подписи выручки — над линией или под ней
for i, (month, revenue) in enumerate(
    zip(monthly_payments["month"], revenue_mln)
):

    # Высота столбца в координатах левой оси
    payment_height = monthly_payments["payment_count"].iloc[i]

    # Переводим высоту столбца в координаты правой оси
    payment_height_on_ax2 = (
        payment_height / ax1.get_ylim()[1]
        * ax2.get_ylim()[1]
    )

    # Если подпись выручки может пересечь подпись платежей,
    # размещаем её под линией
    if abs(revenue - payment_height_on_ax2) < 8:
        offset = -12
        vertical_align = "top"
    else:
        offset = 12
        vertical_align = "bottom"

    ax2.annotate(
        f"{revenue:.1f}",
        xy=(month, revenue),
        xytext=(0, offset),
        textcoords="offset points",
        ha="center",
        va=vertical_align,
        fontsize=8,
        color="darkblue",
        fontweight="bold"
    )

# 4. Средний платёж — зелёные рамочки у основания столбцов
for _, row in monthly_payments.iterrows():
    ax1.annotate(
        f'{row["avg_payment"]:.0f} ₽',
        xy=(row["month"], 0),
        xytext=(0, 12),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8,
        color="darkgreen",
        bbox=dict(
            facecolor="white",
            edgecolor="forestgreen",
            alpha=0.95,
            boxstyle="round,pad=0.25"
        )
    )

# 5. Оформление оси X
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.setp(
    ax1.get_xticklabels(),
    rotation=45,
    ha="right"
)

ax1.set_xlabel("Месяц")

# 6. Заголовок и сетка
ax1.set_title(
    "Динамика выручки, количества успешных платежей и среднего платежа",
    fontsize=13,
    pad=15
)

ax1.grid(axis="y", alpha=0.2)
ax1.set_axisbelow(True)

# 7. Легенда
legend_elements = [
    Patch(
        facecolor="lightsteelblue",
        label="Количество успешных платежей, тыс. шт."
    ),
    Line2D(
        [0], [0],
        color="darkblue",
        marker="o",
        linewidth=2,
        label="Выручка, млн ₽"
    ),
    Patch(
        facecolor="white",
        edgecolor="forestgreen",
        label="Средний платёж, ₽"
    )
]

ax1.legend(
    handles=legend_elements,
    loc="upper left",
    fontsize=9
)

plt.tight_layout()
plt.show()

_С 01.2025 по 01.2026 выручка постепенно увеличивалась, но интересны 4 периода: **02.2026, 03.2026, 04.2026 и 08.2026**. Можно изучить, с чем связано падение выручки в феврале, апреле и августе - связано это с оттоком клиентом, кол-вом дней в месяце или чем-то еще. Так же интересно резкое увеличение в марте - это результат увеличения среднего платежа, притока клиентов, изменения тарификации или чего-то еще._

#### Новые клиенты
Посмотрим, как меняется кол-во новых регистраций.

In [ ]:
monthly_clients = q("""
    SELECT
        date_trunc('month', registered_at)::date AS month,
        COUNT(DISTINCT client_id) AS new_clients
    FROM prime.clients
    GROUP BY date_trunc('month', registered_at)::date
    ORDER BY month
""")

monthly_clients["month"] = pd.to_datetime(
    monthly_clients["month"]
)

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))

# 1. Столбцы — количество новых клиентов
bars = ax.bar(
    monthly_clients["month"].dt.strftime("%Y-%m"),
    monthly_clients["new_clients"],
    color="lightsteelblue",
    alpha=0.7,
    width=0.8
)

# 2. Подписи над столбцами — в тысячах человек
for bar in bars:
    height = bar.get_height()

    ax.annotate(
        f"{height / 1000:.1f}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8,
        color="steelblue",
        fontweight="bold"
    )

# 3. Оформление графика
ax.set_title(
    "Количество новых клиентов ПРАЙМ по месяцам",
    fontsize=13,
    pad=15
)

ax.set_xlabel("Месяц регистрации")
ax.set_ylabel("Количество новых клиентов, чел.")

# Оставляем пространство над столбцами для подписей
ax.set_ylim(0, monthly_clients["new_clients"].max() * 1.15)

ax.tick_params(axis="x", rotation=45)

ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

_В таблице клиентов присутствуют регистрации начиная с 2024 года, тогда как данные об успешных платежах доступны только с 2025 года. Поэтому для совместного анализа клиентской и платёжной активности **целесообразно использовать период начиная с января 2025 года**._

_Количество новых регистраций по месяцам относительно стабильно, однако наблюдаются отдельные снижения, в том числе в феврале 2026 года. Для оценки их значимости необходимо учитывать различную продолжительность месяцев и сравнить среднесуточное количество регистраций._

_По данным заметна сезонность - в феврале кол-во новых регистраций падает (кроме 2024 года, но это мог быть только запуск подписки). Тем не менее можно исследовать изменение динамики новых регистраций **02.2026 (почему падение сильнее, чем в 2025 году) и 08.2026 (в 2025 году не было такого падения - это неполный месяц или что то еще)**_

#### Подписки и отток
Посмотрим на Churn Rate и динамику оттока подписок.

In [ ]:
monthly_subscriptions = q("""

-- Создаем временную таблицу со списком месяцев
    WITH months AS (
        SELECT generate_series(
            DATE '2025-01-01',
            DATE '2026-07-01',
            INTERVAL '1 month'
        )::date AS month
    )

    SELECT
        m.month,

-- Подписки, которые начали действовать в текущем месяце
        COUNT(*) FILTER (
            WHERE s.started_at >= m.month
              AND s.started_at < m.month + INTERVAL '1 month'
        ) AS new_subscriptions,

-- Подписки, которые закончили дйствовать в текущем месяце (тут в том числе могут быть подписки, активированные в данном месяце)
        COUNT(*) FILTER (
            WHERE s.ended_at >= m.month
              AND s.ended_at < m.month + INTERVAL '1 month'
        ) AS ended_subscriptions,

-- Сколько подписок действовало на начало месяца
        COUNT(*) FILTER (
            WHERE s.started_at < m.month
              AND (
                  s.ended_at IS NULL
                  OR s.ended_at >= m.month
              )
        ) AS active_start,

-- Сколько подписок завершиось от начальной базы (без учета подписок, которые начались и закончились в текущем месяце)
        COUNT(*) FILTER (
            WHERE s.started_at < m.month
              AND s.ended_at >= m.month
              AND s.ended_at < m.month + INTERVAL '1 month'
        ) AS churned_from_start

    FROM months m
-- Соединяем кажду строку таблицы month с каждой строкой таблицы subscriptions
    CROSS JOIN prime.subscriptions s

    GROUP BY m.month
    ORDER BY m.month
""")

# Преобразуем столбец month в формат даты в pandas
monthly_subscriptions["month"] = pd.to_datetime(
    monthly_subscriptions["month"]
)

# Churn Rate = Кол-во завершившихся подписок за период / Кол-во активных подписок на начало периода * 100%
monthly_subscriptions["churn_rate"] = (
    monthly_subscriptions["churned_from_start"]
    / monthly_subscriptions["active_start"].replace(0, float("nan"))
    * 100
)

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# 1. Создаём график
fig, ax1 = plt.subplots(figsize=(16, 7))

# Новые подписки за месяц — голубые столбцы
bars = ax1.bar(
    monthly_subscriptions["month"],
    monthly_subscriptions["new_subscriptions"],
    width=20,
    color="lightsteelblue",
    alpha=0.7
)

ax1.set_ylabel("Количество новых подписок, шт.")
ax1.set_ylim(
    0,
    monthly_subscriptions["new_subscriptions"].max() * 1.25
)

# Подписи активных подписок НАД столбцами, тыс. шт.
for bar in bars:
    height = bar.get_height()

    ax1.annotate(
        f"{height / 1000:.1f}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8,
        color="steelblue",
        fontweight="bold"
    )

# 2. Завершившиеся подписки — тёмно-синяя линия
ax2 = ax1.twinx()

ended_thousands = (
    monthly_subscriptions["ended_subscriptions"] / 1000
)

ax2.plot(
    monthly_subscriptions["month"],
    ended_thousands,
    color="darkblue",
    marker="o",
    linewidth=2
)

ax2.set_ylabel("Количество завершившихся подписок, тыс. шт.")
ax2.set_ylim(0, ended_thousands.max() * 1.30)

# Подписи завершившихся подписок НАД линией
for month, ended in zip(
    monthly_subscriptions["month"],
    ended_thousands
):
    ax2.annotate(
        f"{ended:.1f}",
        xy=(month, ended),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8,
        color="darkblue",
        fontweight="bold"
    )

# 3. Коэффициент оттока — зелёные рамочки у основания столбцов
for _, row in monthly_subscriptions.iterrows():
    ax1.annotate(
        f'{row["churn_rate"]:.1f}%',
        xy=(row["month"], 0),
        xytext=(0, 12),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8,
        color="darkgreen",
        bbox=dict(
            facecolor="white",
            edgecolor="forestgreen",
            alpha=0.95,
            boxstyle="round,pad=0.25"
        )
    )

# 4. Оформление оси X
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.setp(
    ax1.get_xticklabels(),
    rotation=45,
    ha="right"
)

ax1.set_xlabel("Месяц")

# 5. Заголовок и сетка
ax1.set_title(
    "Динамика новых и завершившихся подписок и коэффициента оттока",
    fontsize=13,
    pad=15
)

ax1.grid(axis="y", alpha=0.2)
ax1.set_axisbelow(True)

# 6. Легенда
legend_elements = [
    Patch(
        facecolor="lightsteelblue",
        label="Новые подписки за месяц, тыс. шт."
    ),
    Line2D(
        [0], [0],
        color="darkblue",
        marker="o",
        linewidth=2,
        label="Завершившиеся подписки за месяц, тыс. шт."
    ),
    Patch(
        facecolor="white",
        edgecolor="forestgreen",
        label="Коэффициент оттока, %"
    )
]

ax1.legend(
    handles=legend_elements,
    loc="upper left",
    fontsize=9
)

plt.tight_layout()
plt.show()

_В течение исследуемого периода увеличиваются масштабы подписочной базы и абсолютное количество завершений подписок. При этом месячный коэффициент оттока подписок, действовавших на начало месяца, остаётся относительно стабильным — около 4%, хотя в отдельные месяцы наблюдаются колебания._

_Рост абсолютного количества завершений не обязательно свидетельствует об ухудшении удержания: он может быть обусловлен увеличением самой подписочной базы._

_Для дальнейшего анализа представляет интерес разделение **изменения числа завершившихся подписок на эффект размера активной базы и эффект изменения коэффициента оттока**. Дополнительно можно проверить,**различается ли отток между тарифами, пробными и платными подписками и группами клиентов**._

#### Каналы привлечения
Посмотрим, какой канал привлечения приносит большее кол-во клиентов подписки.

In [ ]:
channels = q("""
    SELECT
        COALESCE(acquisition_channel, 'Не указан') AS channel,
        COUNT(DISTINCT client_id) AS clients_count
    FROM prime.clients
    GROUP BY acquisition_channel
    ORDER BY clients_count DESC
""")

channels["share_pct"] = (
    channels["clients_count"]
    / channels["clients_count"].sum()
    * 100
)

In [ ]:
# 1. Контрастная цветовая гамма
colors = [
    "#8CB6D9", 
    "#A3A0D8", 
    "#8CCFB0",  
    "#C5A0DE",  
    "#F4BE91", 
    "#82C7C9",  
    "#E9A4AD"   
]

# 2. Функция для подписей: доля (%) и количество (тыс. чел.)
def format_label(pct):
    total = channels["clients_count"].sum()
    count = pct / 100 * total

    return f"{pct:.1f}%\n{count / 1000:.1f} тыс."

# 3. Создаём круговую диаграмму
fig, ax = plt.subplots(figsize=(8, 4))

wedges, texts, autotexts = ax.pie(
    channels["clients_count"],
    colors=colors,
    startangle=90,
    autopct=format_label,
    pctdistance=0.7,
    radius=0.85,
    wedgeprops={
        "edgecolor": "white",
        "linewidth": 2
    },
    textprops={
        "fontsize": 8,
        "fontweight": "bold"
    }
)

# 4. Оформление
ax.set_title(
    "Структура клиентской базы ПРАЙМ по каналам привлечения",
    fontsize=10,
    pad=10
)

ax.legend(
    wedges,
    channels["channel"],
    title="Канал привлечения",
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    fontsize=8
)

ax.axis("equal")

plt.tight_layout()
plt.show()

_Наибольшую долю зарегистрированных клиентов формируют органический канал (**23,6%**) и performance (**21,6%**). Среди каналов с известным источником привлечения наименьшую долю занимает referral (**7,8%**). Приблизительно у 2% клиентов канал привлечения не указан._

_При этом структура зарегистрированной клиентской базы не позволяет напрямую оценить эффективность каналов: для этого необходимо дополнительно изучить **конверсию в оформление подписки, платёжную активность и расходы на привлечение**._

#### Использование сервисов
Посмотрим, для какие сервисов клиенты чаще всего используют подписку.

In [ ]:
monthly_usage = q("""
    SELECT
        date_trunc('month', u.started_at)::date AS month,
        s.service_name AS service,
        COUNT(*) AS usage_count,
        COUNT(DISTINCT u.client_id) AS active_users

    FROM prime.service_usage u

    LEFT JOIN prime.services s
        ON u.service_id = s.service_id

    GROUP BY
        date_trunc('month', u.started_at)::date,
        s.service_name

    ORDER BY
        month,
        usage_count DESC
""")

monthly_usage["month"] = pd.to_datetime(
    monthly_usage["month"]
)

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# 1. Подготавливаем месячные показатели

monthly_total = (
    monthly_usage
    .groupby("month", as_index=False)["usage_count"]
    .sum()
)

# Общее количество уникальных пользователей нельзя получить
# суммированием active_users по разным сервисам.
# Поэтому для графика считаем пользователей по всем сервисам.

monthly_active = q("""
    SELECT
        date_trunc('month', started_at)::date AS month,
        COUNT(DISTINCT client_id) AS active_users
    FROM prime.service_usage
    GROUP BY date_trunc('month', started_at)::date
    ORDER BY month
""")

monthly_active["month"] = pd.to_datetime(monthly_active["month"])

# Объединяем показатели в одну таблицу
monthly_total = monthly_total.merge(
    monthly_active,
    on="month",
    how="left"
)

# 2. Находим топ-3 сервиса каждого месяца

top3 = (
    monthly_usage
    .sort_values(
        ["month", "usage_count"],
        ascending=[True, False]
    )
    .groupby("month")
    .head(3)
)

top3_labels = (
    top3.groupby("month")["service"]
    .apply(lambda x: "\n".join(
        f"{i}. {name}"
        for i, name in enumerate(x, start=1)
    ))
)

# 3. Создаём график

fig, ax1 = plt.subplots(figsize=(18, 8))

# Количество использований — голубые столбцы
usage_thousands = monthly_total["usage_count"] / 1000

bars = ax1.bar(
    monthly_total["month"],
    usage_thousands,
    width=20,
    color="lightsteelblue",
    alpha=0.7
)

ax1.set_ylabel("Количество использований, тыс.")
ax1.set_ylim(0, usage_thousands.max() * 1.30)

# Подписи количества использований над столбцами
for bar in bars:
    height = bar.get_height()

    ax1.annotate(
        f"{height:.1f}",
        xy=(bar.get_x() + bar.get_width() / 2, height),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8,
        color="steelblue",
        fontweight="bold"
    )

# 4. Активные пользователи — тёмно-синяя линия

ax2 = ax1.twinx()

users_thousands = monthly_total["active_users"] / 1000

ax2.plot(
    monthly_total["month"],
    users_thousands,
    color="darkblue",
    marker="o",
    linewidth=2
)

ax2.set_ylabel("Активные пользователи, тыс. чел.")
ax2.set_ylim(0, users_thousands.max() * 1.30)

# Подписи количества активных пользователей
for month, users in zip(
    monthly_total["month"],
    users_thousands
):
    ax2.annotate(
        f"{users:.1f}",
        xy=(month, users),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=8,
        color="darkblue",
        fontweight="bold"
    )

# 5. Топ-3 сервиса — зелёные рамочки у основания столбцов

for month in monthly_total["month"]:

    label = top3_labels.get(month, "Нет данных")

    ax1.annotate(
        label,
        xy=(month, 0),
        xytext=(0, 12),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=7,
        color="darkgreen",
        bbox=dict(
            facecolor="white",
            edgecolor="forestgreen",
            alpha=0.95,
            boxstyle="round,pad=0.3"
        )
    )

# 6. Оформление оси X

ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

plt.setp(
    ax1.get_xticklabels(),
    rotation=45,
    ha="right"
)

ax1.set_xlabel("Месяц")

# 7. Заголовок и сетка

ax1.set_title(
    "Динамика использования сервисов ПРАЙМ и топ-3 сервиса по месяцам",
    fontsize=13,
    pad=15
)

ax1.grid(axis="y", alpha=0.2)
ax1.set_axisbelow(True)

# 8. Легенда

legend_elements = [
    Patch(
        facecolor="lightsteelblue",
        label="Количество использований, тыс."
    ),
    Line2D(
        [0], [0],
        color="darkblue",
        marker="o",
        linewidth=2,
        label="Активные пользователи, тыс. чел."
    ),
    Patch(
        facecolor="white",
        edgecolor="forestgreen",
        label="Топ-3 сервиса по количеству использований"
    )
]

ax1.legend(
    handles=legend_elements,
    loc="upper left",
    fontsize=9
)

plt.tight_layout()
plt.show()

_Количество использований партнёрских сервисов в целом увеличивается, однако** в феврале и апреле 2026 года наблюдаются локальные снижения**. Аналогичные изменения заметны и в динамике успешных платежей, что позволяет выдвинуть гипотезу о **возможной связи платёжной и пользовательской активности.**_

_Также целесообразно проверить, насколько устойчивы предпочтения пользователей и **связано ли использование отдельных сервисов с продолжением подписки**._

Предварительный обзор выявил несколько потенциально значимых направлений исследования: **факторы изменения выручки, удержание подписчиков, эффективность каналов привлечения и связь пользовательской активности с продолжением подписки.** Для выбора основного бизнес-вопроса необходимо провести целевые проверки качества данных, определить единый период и уровень анализа, а затем сформулировать проверяемую гипотезу и подход к факторной декомпозиции.

### 1.3. Разведочный анализ данных

В начале исследования проведем разведочный анализ данных: проверим их на полноту и сопоставимость.

#### Структура и объем данных

In [ ]:
tables = [
    "clients",
    "subscriptions",
    "payments",
    "service_usage",
    "services",
    "tariffs",
    "marketing_spend"
]

columns_info = q("""
    SELECT
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_schema = 'prime'
      AND table_name IN (
          'clients', 'subscriptions', 'payments',
          'service_usage', 'services', 'tariffs',
          'marketing_spend'
      )
    ORDER BY table_name, ordinal_position
""")

display(columns_info)

In [ ]:
table_sizes = q("""
    SELECT 'clients' AS table_name,
           COUNT(*) AS row_count,
           COUNT(DISTINCT client_id) AS unique_ids
    FROM prime.clients

    UNION ALL

    SELECT 'subscriptions',
           COUNT(*),
           COUNT(DISTINCT subscription_id)
    FROM prime.subscriptions

    UNION ALL

    SELECT 'payments',
           COUNT(*),
           COUNT(DISTINCT payment_id)
    FROM prime.payments

    UNION ALL

    SELECT 'service_usage',
           COUNT(*),
           COUNT(DISTINCT usage_id)
    FROM prime.service_usage

    UNION ALL

    SELECT 'services',
           COUNT(*),
           COUNT(DISTINCT service_id)
    FROM prime.services

    UNION ALL

    SELECT 'marketing_spend',
           COUNT(*),
           COUNT(DISTINCT spend_id)
    FROM prime.marketing_spend

    ORDER BY table_name
""")

table_sizes["duplicate_ids"] = (
    table_sizes["row_count"] - table_sizes["unique_ids"]
)

display(table_sizes)

_Все проверенные ID уникальны._

#### Пропуски

In [ ]:
missing_results = []

for table in tables:

    # Получаем названия столбцов текущей таблицы
    table_columns = columns_info.loc[
        columns_info["table_name"] == table,
        "column_name"
    ].tolist()

    # Для каждого столбца создаём проверку на NULL
    checks = ", ".join(
        f'COUNT(*) FILTER (WHERE "{col}" IS NULL) AS "{col}"'
        for col in table_columns
    )

    # Выполняем запрос
    result = q(f"""
        SELECT
            COUNT(*) AS total_rows,
            {checks}
        FROM prime.{table}
    """)

    total_rows = result.loc[0, "total_rows"]

    # Преобразуем результат в удобную длинную таблицу
    for col in table_columns:
        missing_count = result.loc[0, col]

        missing_results.append({
            "table_name": table,
            "column_name": col,
            "missing_count": missing_count,
            "missing_pct": (
                missing_count / total_rows * 100
                if total_rows > 0 else None
            )
        })

missing_audit = pd.DataFrame(missing_results)

display(
    missing_audit.sort_values(
        ["missing_pct", "table_name"],
        ascending=[False, True]
    ).round({"missing_pct": 2})
)

_У **167978** подписок отсутствуют даты окончания и причины отмены, что согласуется с отсутствием зафиксированного завершения. При расчёте активности на конкретную дату дополнительно учитывается дата начала подписки. У **10000** клиентов (**2%**) нет информации о канале привлечения. В остальных полях таблиц NULL отсутствует._

#### Даты

In [ ]:
date_ranges = q("""
    SELECT
        'clients' AS table_name,
        'registered_at' AS date_column,
        MIN(registered_at) AS min_date,
        MAX(registered_at) AS max_date
    FROM prime.clients

    UNION ALL

    SELECT
        'subscriptions',
        'started_at',
        MIN(started_at),
        MAX(started_at)
    FROM prime.subscriptions

    UNION ALL

    SELECT
        'subscriptions',
        'ended_at',
        MIN(ended_at),
        MAX(ended_at)
    FROM prime.subscriptions

    UNION ALL

    SELECT
        'payments',
        'paid_at',
        MIN(paid_at),
        MAX(paid_at)
    FROM prime.payments

    UNION ALL

    SELECT
        'service_usage',
        'started_at',
        MIN(started_at),
        MAX(started_at)
    FROM prime.service_usage

    UNION ALL

    SELECT
        'marketing_spend',
        'spend_date',
        MIN(spend_date)::timestamptz,
        MAX(spend_date)::timestamptz
    FROM prime.marketing_spend

    ORDER BY table_name, date_column
""")

display(date_ranges)

_Период, доступный для анализа сразу во всех таблицах: **06.01.2025-29.08.2026**._

In [ ]:
event_coverage = q("""
    WITH events AS (
        SELECT
            'clients' AS source,
            registered_at::date AS event_date
        FROM prime.clients

        UNION ALL

        SELECT
            'payments_success',
            paid_at::date
        FROM prime.payments
        WHERE status = 'success'

        UNION ALL

        SELECT
            'service_usage',
            started_at::date
        FROM prime.service_usage
    )

    SELECT
        source,
        date_trunc('month', event_date)::date AS month,
        COUNT(*) AS events_count,
        COUNT(DISTINCT event_date) AS days_with_events

    FROM events
    WHERE event_date IS NOT NULL

    GROUP BY
        source,
        date_trunc('month', event_date)::date

    ORDER BY source, month
""")

event_coverage["month"] = pd.to_datetime(
    event_coverage["month"]
)

event_coverage["days_in_month"] = (
    event_coverage["month"].dt.days_in_month
)

event_coverage["days_without_events"] = (
    event_coverage["days_in_month"]
    - event_coverage["days_with_events"]
)

# Оставляем только месяцы, в которых есть дни без событий
coverage_issues = event_coverage[
    event_coverage["days_without_events"] > 0
].copy()

# Выводим только необходимые столбцы
display(
    coverage_issues[
        ["source", "month", "days_with_events",
         "days_in_month", "days_without_events"]
    ].reset_index(drop=True)
)

_В таблице clients было **8** дней без записей, в payments - **8**, в service_usage - **9**. Само по себе отсутствие событий не доказывает потерю данных, однако эти периоды следует учитывать при сравнении месячных показателей._

In [ ]:
missing_payment_days = q("""
    WITH calendar AS (
        SELECT generate_series(
            DATE '2025-01-01',
            DATE '2026-08-31',
            INTERVAL '1 day'
        )::date AS day
    ),

    payment_days AS (
        SELECT DISTINCT paid_at::date AS day
        FROM prime.payments
        WHERE status = 'success'
    )

    SELECT
        c.day AS day_without_successful_payments

    FROM calendar c

    LEFT JOIN payment_days p
        ON c.day = p.day

    WHERE p.day IS NULL

    ORDER BY c.day
""")

display(missing_payment_days)

_Таблица выводит даты, в которые не было платежей._

In [ ]:
# Проверяем корректность дат и связей между таблицами

data_integrity = q("""
    -- 1. Даты окончания подписок
    SELECT
        'Даты подписок' AS check_name,
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE ended_at < started_at
        ) AS violations
    FROM prime.subscriptions

    UNION ALL

    -- 2. Даты использования сервисов
    SELECT
        'Даты использования сервисов',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE ended_at < started_at
        )
    FROM prime.service_usage

    UNION ALL

    -- 3. Подписки, начавшиеся раньше регистрации клиента
    SELECT
        'Подписка раньше регистрации',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE c.client_id IS NOT NULL
              AND s.started_at < c.registered_at
        )
    FROM prime.subscriptions s
    LEFT JOIN prime.clients c
        ON s.owner_client_id = c.client_id

    UNION ALL

    -- 4. Подписки без существующего клиента
    SELECT
        'Подписки без клиента',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE c.client_id IS NULL
        )
    FROM prime.subscriptions s
    LEFT JOIN prime.clients c
        ON s.owner_client_id = c.client_id

    UNION ALL

    -- 5. Платежи без существующей подписки
    SELECT
        'Платежи без подписки',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE s.subscription_id IS NULL
        )
    FROM prime.payments p
    LEFT JOIN prime.subscriptions s
        ON p.subscription_id = s.subscription_id

    UNION ALL

    -- 6. Использования без существующего клиента
    SELECT
        'Использования без клиента',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE c.client_id IS NULL
        )
    FROM prime.service_usage u
    LEFT JOIN prime.clients c
        ON u.client_id = c.client_id

    UNION ALL

    -- 7. Использования без существующего сервиса
    SELECT
        'Использования без сервиса',
        COUNT(*),
        COUNT(*) FILTER (
            WHERE s.service_id IS NULL
        )
    FROM prime.service_usage u
    LEFT JOIN prime.services s
        ON u.service_id = s.service_id
""")

# Рассчитываем долю нарушений
data_integrity["violations_pct"] = (
    data_integrity["violations"]
    / data_integrity["total_rows"].replace(0, float("nan"))
    * 100
)

# Добавляем результат проверки
data_integrity["status"] = data_integrity["violations"].apply(
    lambda x: "OK" if x == 0 else "Требует проверки"
)

# Выводим итоговую таблицу
display(
    data_integrity.style.format({
        "total_rows": "{:,.0f}",
        "violations": "{:,.0f}",
        "violations_pct": "{:.2f}%"
    }, na_rep="—")
)

_Нарушений бизнес-логики по временному признаку не было обнаружено, кроме **15947 случаев использования сервисов, в которых дата окончания раньше, чем дата начала**. В подписках такого нет, случаев подписки раньше регистрации, подписки без клиента, платежей без подписки, пользования без клиента и пользования без сервиса тоже нет._

#### Бизнес-логика

In [ ]:
# Проверяем историю тарифных цен и пересечения периодов

tariff_check = q("""
    WITH tariff_versions AS (
        SELECT
            tariff_id,
            COUNT(*) AS versions_count,
            MIN(valid_from) AS first_valid_from,
            MAX(valid_to) AS last_valid_to
        FROM prime.tariffs
        GROUP BY tariff_id
    ),

    tariff_overlaps AS (
        SELECT
            t1.tariff_id,
            COUNT(*) AS overlaps_count
        FROM prime.tariffs t1

        JOIN prime.tariffs t2
            ON t1.tariff_id = t2.tariff_id
           AND t1.valid_from < t2.valid_from

        WHERE t1.valid_to IS NULL
           OR t1.valid_to > t2.valid_from

        GROUP BY t1.tariff_id
    )

    SELECT
        v.tariff_id,
        v.versions_count,
        v.first_valid_from,
        v.last_valid_to,
        COALESCE(o.overlaps_count, 0) AS overlaps_count

    FROM tariff_versions v

    LEFT JOIN tariff_overlaps o
        ON v.tariff_id = o.tariff_id

    ORDER BY v.tariff_id
""")

# Добавляем результат проверки
tariff_check["status"] = tariff_check["overlaps_count"].apply(
    lambda x: "OK" if x == 0 else "Требует проверки"
)

display(tariff_check)

_У тарифа 1 было 2 версии цены, у тарифа 2 - три версии, причем **во 2 тарифе есть 2 периода, в которые действовала разная цена одновременно**. Такие случаи требуют дополнительной проверки._

In [ ]:
payment_quality = q("""
    SELECT
        COALESCE(status, 'NULL') AS status,

        COUNT(*) AS payment_count,

        COUNT(*) FILTER (
            WHERE amount IS NULL
        ) AS missing_amount,

        COUNT(*) FILTER (
            WHERE amount = 0
        ) AS zero_amount,

        COUNT(*) FILTER (
            WHERE amount < 0
        ) AS negative_amount,

        MIN(amount) AS min_amount,
        MAX(amount) AS max_amount

    FROM prime.payments

    GROUP BY status
    ORDER BY payment_count DESC
""")

display(payment_quality)

_Таблица выводит основную информацию по каждому статусу платежей - аномалий нет._

In [ ]:
subscription_categories = q("""
    SELECT
        tariff_id,
        is_trial,
        COUNT(*) AS subscriptions_count

    FROM prime.subscriptions

    GROUP BY tariff_id, is_trial

    ORDER BY tariff_id, is_trial
""")

display(subscription_categories)

_В 1 тарифе было **35441** подписок по пробному периоду, во 2 - **21343**. Это важно учесть для дальнейшего анализа клиентской базы._

In [ ]:
cancel_reasons = q("""
    SELECT
        COALESCE(cancel_reason, 'Не указана') AS cancel_reason,
        COUNT(*) AS subscriptions_count

    FROM prime.subscriptions

    GROUP BY cancel_reason

    ORDER BY subscriptions_count DESC
""")

display(cancel_reasons)

_Наиболее распространённые зарегистрированные причины завершения — **not_using** (56 416) и **too_expensive** (40 050). Вместе они составляют около 60% завершений, что обосновывает дальнейшее исследование структуры причин по тарифам и месяцам._

#### Вывод

Проверка качества данных показала, что основные таблицы ПРАЙМ в целом пригодны для дальнейшего анализа. В исследованных таблицах не обнаружены повторяющиеся идентификаторы, а связи между клиентами, подписками, платежами и использованием сервисов не содержат нарушений.

**Полнота данных.** Пропуски сосредоточены преимущественно в полях `ended_at` и `cancel_reason` (50,31% подписок), что согласуется с отсутствием зафиксированного завершения подписки. Канал привлечения не указан у 2% клиентов. В остальных проверенных полях пропуски отсутствуют.

**Временные ограничения.** Регистрации и подписки доступны с января 2024 года, тогда как платежи, использование сервисов и маркетинговые расходы — с января 2025 года. Для сопоставимого анализа этих источников предпочтителен период с февраля 2025 по июль 2026 года.

**Обнаруженные аномалии.** В таблице использования сервисов выявлено 15 947 событий (около 0,20%) с датой окончания раньше даты начала. Кроме того, в истории цен тарифа № 2 обнаружены пересекающиеся периоды действия тарифных условий. Эти особенности необходимо учитывать при расчёте длительности сессий и анализе влияния тарифных цен.

### 1.4. Бизнес-задача

#### Бизнес-контекст и актуальность

Предварительный обзор ПРАЙМ показал, что на фоне роста активной подписочной базы увеличивается и абсолютное количество завершившихся подписок. При этом месячный коэффициент оттока подписок, действовавших на начало месяца, остаётся относительно стабильным — около 4%.

На графиках 2026 года также заметно изменение соотношения между количеством новых и завершившихся подписок. Это вызывает вопрос об устойчивости дальнейшего роста подписочной базы: даже при стабильном коэффициенте оттока увеличение числа завершений может сокращать чистый прирост подписок.

Проведённый аудит подтвердил пригодность данных о подписках для исследования: идентификаторы уникальны, даты начала и завершения логически согласованы, связи с владельцами не содержат нарушений.

#### Бизнес-вопрос

**Почему с января по июль 2026 года количество завершившихся подписок увеличилось сильнее количества новых подписок и какие факторы определили сокращение месячного чистого прироста подписочной базы?**

#### Цель исследования

Оценить динамику новых и завершившихся подписок, определить факторы изменения числа завершений и чистого прироста подписочной базы, сравнить показатели оттока по тарифам и исследовать структуру причин завершения.

#### Гипотезы

**H1.** Увеличение абсолютного числа завершений преимущественно связано с ростом активной подписочной базы, а не с существенным ухудшением коэффициента оттока.

**H2.** Сокращение месячного чистого прироста подписочной базы с января по июль 2026 года преимущественно связано с увеличением количества завершений, тогда как изменение притока новых подписок оказывает меньшее влияние.

**H3.** Коэффициенты оттока различаются между тарифами, а структура причин завершения, включая not_using и too_expensive, может различаться по тарифам и периодам.

## Часть 2. Анализ метрик

### 2.1. Выбор зерна, детализации, периода и сегментов

#### Зерно данных

**Основное зерно аналитической витрины — подписка (`subscription_id`).** Каждая строка соответствует одной подписке и содержит сведения о её владельце, тарифе, дате начала, дате завершения и причине отмены.

Такой выбор обусловлен бизнес-вопросом исследования, который связан с динамикой новых и завершившихся подписок ПРАЙМ. В отличие от зерна «клиент», зерно «подписка» позволяет учитывать несколько подписок одного клиента независимо друг от друга. Зерно «платёж» не подходит для основной задачи, поскольку одна подписка может иметь несколько платежей, а завершение подписки не обязательно совпадает с датой платежа.

#### Уровень детализации

**Основная временная детализация — календарный месяц.**

Месячная агрегация позволяет сопоставлять количество новых и завершившихся подписок, рассчитывать чистый прирост подписочной базы и коэффициент оттока, а также выявлять устойчивые изменения показателей в течение 2026 года.

#### Период исследования

**Январь–июль 2026 года.**

Выбор связан с предварительно обнаруженным (см. раздел  1.2) изменением соотношения между количеством новых и завершившихся подписок в 2026 году. Анализ семи последовательных месяцев позволит проверить, является ли это изменение устойчивым и как оно отражается на чистом приросте подписочной базы.

Август 2026 года исключается, поскольку доступные данные заканчиваются до завершения календарного месяца.

#### Сегменты анализа

**Основной сегмент — тариф (`tariff_id`).**

Сегментация по тарифам позволяет проверить гипотезу о различиях в коэффициентах оттока, оценить вклад каждого тарифа в изменение числа завершений и определить, в каких тарифных группах возникают основные риски для роста подписочной базы.

Дополнительно при исследовании завершений будет использоваться **причина отмены (`cancel_reason`)**. Это позволит сравнить структуру зарегистрированных причин завершения, включая `not_using` и `too_expensive`, по тарифам и периодам.

Сегментация по каналам привлечения не включается в основной анализ, поскольку основное внимание уделяется жизненному циклу подписок, а не эффективности привлечения клиентов.

### 2.2. Витрина
Витрина построена на основе таблиц **prime.subscriptions** (зерно - одна подписка) и **prime.clients** (дополнительные данные о клиентах).

Витрина сохраняет полную историю подписок, включая записи, созданные до 2026 года. Это необходимо для последующего расчёта активной базы на начало каждого исследуемого месяца.

In [ ]:
# Загружаем подписки — основа витрины
subs = q("""
    SELECT
        subscription_id,
        owner_client_id,
        tariff_id,
        started_at,
        ended_at,
        is_trial,
        cancel_reason
    FROM prime.subscriptions
""")

# Загружаем данные о владельцах
clients_for_mart = q("""
    SELECT
        client_id,
        registered_at,
        acquisition_channel,
        region_code
    FROM prime.clients
""")

# Создаём витрину: одна строка = одна подписка
subscription_mart = subs.merge(
    clients_for_mart,
    left_on="owner_client_id",
    right_on="client_id",
#left, чтобы сохранить все подписки, даже если для отдельных записей не найден клиент
    how="left",
#потому что несколько подписок может принадлежать одному клиенту
    validate="many_to_one"
)

display(subscription_mart.head())

In [ ]:
# Проверяем, сохранилось ли зерно "подписка"

mart_check = pd.DataFrame({
    "Проверка": [
        "Строк в исходной таблице",
        "Строк в витрине",
        "Уникальных подписок в витрине",
        "Дублирующихся subscription_id",
        "Подписок без найденного клиента"
    ],
    "Значение": [
        len(subs),
        len(subscription_mart),
        subscription_mart["subscription_id"].nunique(),
        subscription_mart["subscription_id"].duplicated().sum(),
        subscription_mart["client_id"].isna().sum()
    ]
})

display(mart_check)

# Автоматическая проверка условий
assert len(subscription_mart) == len(subs), \
    "Ошибка: количество строк изменилось после соединения!"

assert subscription_mart["subscription_id"].is_unique, \
    "Ошибка: зерно подписки нарушено, появились дубликаты!"

assert subscription_mart["client_id"].notna().all(), \
    "Ошибка: для некоторых подписок не найден владелец!"

print("Все проверки пройдены")

_Витрина содержит 333 905 строк — столько же, сколько исходная таблица подписок. Все subscription_id уникальны, дублирования и потери строк не обнаружены, для каждой подписки найден владелец. Следовательно, зерно «подписка» сохранено, соединение выполнено корректно._

### 2.3. Метрики

In [ ]:
# настройка периода и библиотек
import seaborn as sns

sns.set_theme(style="whitegrid", palette="pastel")

# Работаем с копией проверенной витрины
mart = subscription_mart.copy()

# Приводим даты к единому формату
for col in ["started_at", "ended_at"]:
    mart[col] = pd.to_datetime(mart[col], utc=True).dt.tz_localize(None)

# Исследуем январь–июль 2026 года
months = pd.date_range("2026-01-01", "2026-07-01", freq="MS")

# Подписи для графиков
month_labels = [m.strftime("%m.%Y") for m in months]

#### Новые и завершившиеся подписки
Ответ на вопрос: Действительно ли количество завершившихся подписок увеличивается быстрее, чем количество новых?

In [ ]:
# Создаём таблицу с месяцами исследования
subscriptions_dynamics = pd.DataFrame({"month": months})

# Рассчитываем количество новых подписок по месяцам
subscriptions_dynamics["new_subscriptions"] = [
    (
        # Дата начала подписки не раньше начала месяца
        (mart["started_at"] >= month)

        # Дата начала подписки раньше начала следующего месяца
        & (mart["started_at"] < month + pd.DateOffset(months=1))
    ).sum()  # True считается как 1, поэтому sum() даёт число подписок
    for month in months
]

# Рассчитываем количество завершившихся подписок по месяцам
subscriptions_dynamics["ended_subscriptions"] = [
    (
        # Дата завершения не раньше начала месяца
        (mart["ended_at"] >= month)

        # Дата завершения раньше начала следующего месяца
        & (mart["ended_at"] < month + pd.DateOffset(months=1))
    ).sum()
    for month in months
]

# Рассчитываем изменение числа новых подписок относительно января
subscriptions_dynamics["new_change_pct"] = (
    subscriptions_dynamics["new_subscriptions"]

    # iloc[0] — количество новых подписок в январе
    / subscriptions_dynamics["new_subscriptions"].iloc[0]

    # Вычитаем 1, чтобы получить относительное изменение
    - 1
) * 100  # Переводим результат в проценты

# Аналогично рассчитываем изменение числа завершений
subscriptions_dynamics["ended_change_pct"] = (
    subscriptions_dynamics["ended_subscriptions"]
    / subscriptions_dynamics["ended_subscriptions"].iloc[0]
    - 1
) * 100

# Выводим одну таблицу со всеми четырьмя показателями
# Округляем только процентные столбцы
display(
    subscriptions_dynamics.round({
        "new_change_pct": 2,
        "ended_change_pct": 2
    })
)

In [ ]:
# график
# Подготавливаем данные для графика абсолютного количества подписок
# Преобразуем таблицу в длинный формат для seaborn
absolute_plot = subscriptions_dynamics.melt(
    id_vars="month",
    value_vars=["new_subscriptions", "ended_subscriptions"],
    var_name="metric",
    value_name="value"
)

# Подготавливаем данные для графика относительного изменения
relative_plot = subscriptions_dynamics.melt(
    id_vars="month",
    value_vars=["new_change_pct", "ended_change_pct"],
    var_name="metric",
    value_name="value"
)

# Заменяем технические названия метрик на понятные подписи
metric_names = {
    "new_subscriptions": "Новые",
    "ended_subscriptions": "Завершившиеся",
    "new_change_pct": "Новые",
    "ended_change_pct": "Завершившиеся"
}

# Применяем понятные названия к обеим таблицам
absolute_plot["metric"] = absolute_plot["metric"].replace(metric_names)
relative_plot["metric"] = relative_plot["metric"].replace(metric_names)

# Задаём одинаковые цвета метрик на обеих панелях
colors = {
    "Новые": "steelblue",
    "Завершившиеся": "indianred"
}

# Создаём один рисунок с двумя панелями
fig, axes = plt.subplots(1, 2, figsize=(18, 6))


# --------------------------------------------------
# ЛЕВАЯ ПАНЕЛЬ: абсолютное количество подписок
# --------------------------------------------------

# Строим линии новых и завершившихся подписок
sns.lineplot(
    data=absolute_plot,
    x="month",
    y="value",
    hue="metric",
    palette=colors,
    marker="o",
    linewidth=2.5,
    ax=axes[0]
)

# Добавляем подпись к каждой точке
for _, row in absolute_plot.iterrows():

    # Подписи новых подписок располагаем выше линии,
    # а завершившихся — ниже
    offset = 12 if row["metric"] == "Новые" else -18

    axes[0].annotate(
        # Форматируем количество с пробелами между тысячами
        f'{row["value"]:,.0f}'.replace(",", " "),

        # Координаты подписываемой точки
        xy=(row["month"], row["value"]),

        # Смещение подписи относительно точки
        xytext=(0, offset),
        textcoords="offset points",

        # Оформление подписи
        ha="center",
        fontsize=9,
        color=colors[row["metric"]]
    )

# Добавляем название и подписи осей
axes[0].set_title("Количество новых и завершившихся подписок")
axes[0].set_xlabel("Месяц")
axes[0].set_ylabel("Количество подписок, шт.")

# Подписываем все месяцы
axes[0].set_xticks(months)
axes[0].set_xticklabels(month_labels, rotation=45)

# Оставляем легенду
axes[0].legend(title="")

# Добавляем пространство для подписей
axes[0].margins(x=0.07, y=0.18)


# --------------------------------------------------
# ПРАВАЯ ПАНЕЛЬ: изменение относительно января
# --------------------------------------------------

# Строим линии относительного изменения двух метрик
sns.lineplot(
    data=relative_plot,
    x="month",
    y="value",
    hue="metric",
    palette=colors,
    marker="o",
    linewidth=2.5,
    ax=axes[1]
)

# Добавляем подпись к каждой точке
for _, row in relative_plot.iterrows():

    # Подписи новых подписок располагаем ниже линии,
    # а завершившихся — выше
    offset = -18 if row["metric"] == "Новые" else 12

    axes[1].annotate(
        # Выводим изменение с двумя знаками после запятой
        f'{row["value"]:+.2f}%'.replace(".", ","),

        # Координаты точки
        xy=(row["month"], row["value"]),

        # Смещение подписи
        xytext=(0, offset),
        textcoords="offset points",

        # Оформление подписи
        ha="center",
        fontsize=9,
        color=colors[row["metric"]]
    )

# Добавляем горизонтальную линию нулевого изменения
axes[1].axhline(
    0,
    color="gray",
    linestyle="--",
    linewidth=1
)

# Добавляем название и подписи осей
axes[1].set_title("Изменение относительно января 2026 года")
axes[1].set_xlabel("Месяц")
axes[1].set_ylabel("Изменение, %")

# Подписываем все месяцы
axes[1].set_xticks(months)
axes[1].set_xticklabels(month_labels, rotation=45)

# Убираем повторную легенду
axes[1].get_legend().remove()

# Оставляем место для подписей и стрелки
axes[1].margins(x=0.10, y=0.30)


# --------------------------------------------------
# СТРЕЛКА: выделяем главный аналитический результат
# --------------------------------------------------

# Получаем изменение завершившихся подписок в июле
july_ended_pct = subscriptions_dynamics["ended_change_pct"].iloc[-1]

# Формируем подпись из рассчитанного значения
# Это позволяет не вводить 12,1% вручную
arrow_label = (
    f"Завершения +{july_ended_pct:.1f}%"
    .replace(".", ",")
)

# Добавляем стрелку к июльской точке завершений
axes[1].annotate(
    arrow_label,

    # Стрелка указывает на июльское значение
    xy=(months[-1], july_ended_pct),

    # Текст размещаем левее и выше точки
    xytext=(-105, 60),
    textcoords="offset points",

    # Оформление текста
    fontsize=10,
    fontweight="bold",
    color="indianred",

    # Оформление стрелки
    arrowprops={
        "arrowstyle": "->",
        "color": "indianred",
        "lw": 1.5
    }
)


# --------------------------------------------------
# ОБЩИЙ ЗАГОЛОВОК И ОФОРМЛЕНИЕ
# --------------------------------------------------

# Получаем изменения обеих метрик в июле
new_pct = subscriptions_dynamics["new_change_pct"].iloc[-1]
ended_pct = subscriptions_dynamics["ended_change_pct"].iloc[-1]

# Создаём заголовок с главным выводом
# Значения автоматически берутся из таблицы
main_title = (
    f"Завершения выросли на {ended_pct:.1f}%, "
    f"новые подписки — на {new_pct:.1f}%"
).replace(".", ",")

# Добавляем общий заголовок рисунка
fig.suptitle(
    main_title,
    fontsize=14,
    fontweight="bold"
)

# Корректируем расположение панелей
plt.tight_layout(rect=[0, 0, 1, 0.94])

# Отображаем рисунок
plt.show()

_В январе–июле 2026 года количество новых подписок колебалось около 11 тыс. в месяц: 11 326 в январе и 11 338 в июле (**+0,1%**). Количество завершившихся подписок за тот же период увеличилось с 6 607 до 7 407 (**+12,1%**)._

_Таким образом, **подтверждается исходное наблюдение о более быстром увеличении завершений относительно новых подписок**. При этом динамика не была равномерной: в отдельные месяцы наблюдались колебания обоих показателей._

#### Чистый прирост подписочной базы
Ответ на вопрос: Как различия в динамике новых и завершившихся подписок влияют на рост бизнеса?

In [ ]:
# Создаём отдельную таблицу для анализа чистого прироста
# Используем рассчитанные в 2.3.1 значения новых и завершений
net_growth = subscriptions_dynamics[
    ["month", "new_subscriptions", "ended_subscriptions"]
].copy()

# Рассчитываем чистый прирост за каждый месяц
# Если новых больше завершений, прирост положительный
# Если завершений больше новых, прирост отрицательный
net_growth["net_growth"] = (
    net_growth["new_subscriptions"]
    - net_growth["ended_subscriptions"]
)

# Выводим значения притока, завершений и чистого прироста
display(net_growth)

In [ ]:
# график
fig, ax = plt.subplots(figsize=(11, 5))

# Строим столбчатую диаграмму чистого прироста
sns.barplot(
    data=net_growth,
    x="month",
    y="net_growth",
    color="lightsteelblue",
    ax=ax
)

# Перебираем столбцы диаграммы
for container in ax.containers:

    # Подписываем каждый столбец его фактическим значением
    ax.bar_label(
        container,
        labels=[
            f"{value:,.0f}".replace(",", " ")
            for value in net_growth["net_growth"]
        ],
        padding=4,
        fontsize=10
    )

# Добавляем горизонтальную линию нуля
ax.axhline(0, color="gray", linewidth=1)

# Добавляем название и подписи осей
ax.set_title("Чистый прирост подписочной базы, 2026")
ax.set_xlabel("Месяц")
ax.set_ylabel("Количество подписок")

# Устанавливаем подписи семи месяцев
ax.set_xticks(range(len(months)))
ax.set_xticklabels(month_labels)

# Оставляем место для подписей над столбцами
ax.margins(y=0.15)

# Определяем чистый прирост июля
july_growth = net_growth["net_growth"].iloc[-1]

# Добавляем стрелку к июльскому столбцу
ax.annotate(
    "−16,7% к январю",
    xy=(len(months) - 1, july_growth),
    xytext=(-90, 55),
    textcoords="offset points",
    fontsize=10,
    fontweight="bold",
    color="indianred",
    arrowprops={
        "arrowstyle": "->",
        "color": "indianred",
        "lw": 1.5
    }
)

# Заголовок отражает основной результат
ax.set_title(
    "Месячный чистый прирост сократился на 16,7%",
    fontsize=14,
    fontweight="bold"
)

# Оставляем дополнительное пространство для стрелки
ax.margins(y=0.25)

plt.tight_layout()
plt.show()

Во все рассматриваемые месяцы количество новых подписок превышало число завершений, поэтому чистый прирост оставался положительным. Однако с января по июль 2026 года месячный чистый прирост **замедлился с 4 719 до 3 931 подписки**.

Увеличение завершений относительно новых подписок приводит к уменьшению месячного чистого прироста. Несмотря на продолжающееся расширение базы, её ежемесячный прирост к июлю стал меньше.

**H2 подтверждается** на уровне наблюдаемой динамики и арифметической связи показателей. Для оценки вклада изменения притока и завершений потребуется факторная декомпозиция.

#### Соотношение завершенных и новых подписок
Ответ на вопрос: Насколько больше или меньше завершения по отношению к текущему притоку новых подписок?

In [ ]:
# Создаём отдельную таблицу для расчёта соотношения
ended_to_new = subscriptions_dynamics[
    ["month", "new_subscriptions", "ended_subscriptions"]
].copy()

# Рассчитываем количество завершений относительно новых подписок
ended_to_new["ended_to_new_pct"] = (
    ended_to_new["ended_subscriptions"]

    # Заменяем нули на NaN, чтобы избежать деления на ноль
    / ended_to_new["new_subscriptions"].replace(0, float("nan"))

    # Переводим отношение в проценты
    * 100
)

# Выводим рассчитанное соотношение по месяцам
# Округляем процентный показатель до двух знаков
display(
    ended_to_new[
        ["month", "ended_to_new_pct"]
    ].round({"ended_to_new_pct": 2})
)

Соотношение завершений и новых подписок увеличилось с 58,33% в январе до 65,33% в июле 2026 года (+7,0 п.п.).

Это означает, что в июле на каждые 100 новых подписок приходилось около 65 завершений против 58 в январе. Показатель оставался ниже 100%, поэтому приток продолжал превышать завершения.

**Увеличение соотношения подтверждает, что завершения занимают всё большую долю относительно текущего притока** при сравнении начала и конца периода.

#### Активная подписочная база на начало месяца
Ответ на вопрос: Может ли увеличение числа завершений объясняться тем, что подписочная база стала больше?

In [ ]:
# Создаём отдельную таблицу с месяцами исследования
active_base = pd.DataFrame({"month": months})

# Для каждого месяца рассчитываем количество активных подписок
active_base["active_start"] = [
    (
        # Подписка должна начаться до первого дня месяца
        (mart["started_at"] < month)

        & (
            # Если дата завершения отсутствует,
            # подписка считается незавершённой
            mart["ended_at"].isna()

            # Либо дата завершения не раньше начала месяца
            | (mart["ended_at"] >= month)
        )
    ).sum()  # Суммируем подходящие подписки
    for month in months
]

# Выводим активную базу на начало каждого месяца
display(active_base)

Активная подписочная база на начало месяца увеличилась со 163 809 подписок в январе до 188 410 в июле 2026 года.

Рост активной базы означает, что увеличилось количество подписок, которые потенциально могли завершиться в течение месяца. Следовательно, абсолютное число завершений могло увеличиваться даже без ухудшения удержания.

**Выявлен возможный фактор роста завершений — расширение активной базы.**

#### Churn Rate
Ответ на вопрос: Увеличивается ли количество завершений потому, что подписчики стали чаще прекращать подписку, или потому, что активная база стала больше?

In [ ]:
# Создаём отдельную таблицу на основе рассчитанной активной базы
churn_metrics = active_base.copy()

# Считаем завершения среди подписок,
# которые уже были активны на начало месяца
churn_metrics["churned_from_start"] = [
    (
        # Подписка началась до рассматриваемого месяца
        (mart["started_at"] < month)

        # Дата завершения не раньше начала месяца
        & (mart["ended_at"] >= month)

        # Дата завершения раньше начала следующего месяца
        & (mart["ended_at"] < month + pd.DateOffset(months=1))
    ).sum()
    for month in months
]

# Рассчитываем коэффициент оттока в процентах
churn_metrics["churn_rate"] = (
    churn_metrics["churned_from_start"]

    # Делим на количество активных подписок на начало месяца
    # Нулевые значения заменяем на NaN во избежание деления на ноль
    / churn_metrics["active_start"].replace(0, float("nan"))

    # Переводим долю в проценты
    * 100
)

# Выводим размер базы, количество завершений и Churn Rate
# Округляем только процентный показатель
display(churn_metrics.round({"churn_rate": 2}))

In [ ]:
# график
# Создаём один рисунок с двумя панелями, расположенными рядом
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# ЛЕВАЯ ПАНЕЛЬ: активная подписочная база на начало месяца

# Строим динамику количества активных подписок
sns.lineplot(
    data=active_base,
    x="month",
    y="active_start",
    marker="o",
    linewidth=2.5,
    color="steelblue",
    ax=axes[0]
)

# Добавляем подпись к каждой точке графика
for _, row in active_base.iterrows():

    axes[0].annotate(
        # Выводим целое число с пробелом между тысячами
        f'{row["active_start"]:,.0f}'.replace(",", " "),

        # Указываем координаты точки, которую подписываем
        xy=(row["month"], row["active_start"]),

        # Размещаем подпись на 12 пунктов выше точки
        xytext=(0, 12),
        textcoords="offset points",

        # Выравниваем подпись по центру
        ha="center",
        fontsize=10,
        color="steelblue"
    )

# Добавляем название и подписи осей левой панели
axes[0].set_title("Активная подписочная база на начало месяца")
axes[0].set_xlabel("Месяц")
axes[0].set_ylabel("Количество активных подписок")

# Подписываем все месяцы
axes[0].set_xticks(months)
axes[0].set_xticklabels(month_labels, rotation=45)

# Добавляем пространство над точками для подписей
axes[0].margins(y=0.20)


# ПРАВАЯ ПАНЕЛЬ: коэффициент оттока (Churn Rate)

# Строим динамику коэффициента оттока
sns.lineplot(
    data=churn_metrics,
    x="month",
    y="churn_rate",
    marker="o",
    linewidth=2.5,
    color="indianred",
    ax=axes[1]
)

# Добавляем подпись к каждой точке графика
for _, row in churn_metrics.iterrows():

    axes[1].annotate(
        # Выводим значение с двумя знаками после запятой и символом %
        f'{row["churn_rate"]:.2f}%'.replace(".", ","),

        # Указываем координаты точки
        xy=(row["month"], row["churn_rate"]),

        # Размещаем подпись выше точки
        xytext=(0, 12),
        textcoords="offset points",

        # Выравниваем подпись по центру
        ha="center",
        fontsize=10,
        color="indianred"
    )

# Добавляем название и подписи осей правой панели
axes[1].set_title("Коэффициент оттока (Churn Rate)")
axes[1].set_xlabel("Месяц")
axes[1].set_ylabel("Churn Rate, %")

# Подписываем все месяцы
axes[1].set_xticks(months)
axes[1].set_xticklabels(month_labels, rotation=45)

# Добавляем пространство над точками для подписей
axes[1].margins(y=0.30)

# Добавляем общий заголовок рисунка
fig.suptitle(
    "Динамика активной подписочной базы и коэффициента оттока, 2026",
    fontsize=14
)

# Получаем размер активной базы на начало июля
july_base = active_base["active_start"].iloc[-1]

# Добавляем стрелку к июльскому значению активной базы
axes[0].annotate(
    "Рост базы +15,0%",
    xy=(months[-1], july_base),
    xytext=(-115, -65),
    textcoords="offset points",
    fontsize=10,
    fontweight="bold",
    color="steelblue",
    arrowprops={
        "arrowstyle": "->",
        "color": "steelblue",
        "lw": 1.5
    }
)

# Получаем коэффициент оттока июля
july_churn = churn_metrics["churn_rate"].iloc[-1]

# Добавляем стрелку к июльскому Churn Rate
axes[1].annotate(
    "Churn Rate не вырос",
    xy=(months[-1], july_churn),
    xytext=(-125, -65),
    textcoords="offset points",
    fontsize=10,
    fontweight="bold",
    color="indianred",
    arrowprops={
        "arrowstyle": "->",
        "color": "indianred",
        "lw": 1.5
    }
)

# Общий заголовок сообщает результат сравнения
fig.suptitle(
    "Активная база выросла на 15%, Churn Rate не увеличился",
    fontsize=14,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

В январе–июле 2026 года коэффициент оттока оставался относительно стабильным — около 4%. В январе Churn Rate составлял 4,02%, а в июле — 3,92%.

Следовательно, увеличение абсолютного количества завершений не сопровождалось ростом интенсивности оттока. Напротив, между январём и июлем доля завершившихся подписок в начальной базе немного снизилась.

Рост числа завершений нельзя объяснить увеличением общего Churn Rate между январём и июлем. **Наиболее вероятным объяснением становится расширение активной базы.** H1 предварительно поддерживается данными. Чтобы доказать, какой фактор внёс больший вклад, необходимо выполнить количественную декомпозицию изменения числа завершений.

#### Сегментация по тарифам
Ответ на вопрос: Есть ли тариф, в котором подписки завершаются чаще, и может ли он объяснять увеличение общего числа завершений?

In [ ]:
# Создаём пустой список для результатов каждого месяца
tariff_parts = []

# Последовательно рассматриваем каждый месяц исследования
for month in months:

    # Определяем первый день следующего месяца
    next_month = month + pd.DateOffset(months=1)

    # Определяем подписки, активные на начало месяца
    active_mask = (
        # Подписка началась до рассматриваемого месяца
        (mart["started_at"] < month)

        & (
            # Подписка ещё не завершилась
            mart["ended_at"].isna()

            # Или завершилась не раньше начала месяца
            | (mart["ended_at"] >= month)
        )
    )

    # Определяем завершения среди подписок,
    # которые были активны на начало месяца
    churn_mask = (
        (mart["started_at"] < month)
        & (mart["ended_at"] >= month)
        & (mart["ended_at"] < next_month)
    )

    # Создаём временную таблицу с идентификаторами тарифов
    tariff_data = mart[["tariff_id"]].copy()

    # Преобразуем True/False в 1/0
    # Это позволяет считать подписки через сумму
    tariff_data["active_start"] = active_mask.astype(int)
    tariff_data["churned_from_start"] = churn_mask.astype(int)

    # Группируем данные по тарифам
    # Суммируем активные подписки и завершения
    tariff_result = (
        tariff_data
        .groupby("tariff_id", as_index=False)[
            ["active_start", "churned_from_start"]
        ]
        .sum()
    )

    # Добавляем месяц к рассчитанным показателям
    tariff_result["month"] = month

    # Сохраняем результат месяца
    tariff_parts.append(tariff_result)

# Объединяем результаты всех месяцев
tariff_churn = pd.concat(
    tariff_parts,
    ignore_index=True
)

# Рассчитываем Churn Rate для каждого тарифа и месяца
tariff_churn["churn_rate"] = (
    tariff_churn["churned_from_start"]
    / tariff_churn["active_start"].replace(0, float("nan"))
    * 100
)

# Выводим таблицу с результатами
display(
    tariff_churn[
        [
            "month",
            "tariff_id",
            "active_start",
            "churned_from_start",
            "churn_rate"
        ]
    ].round({"churn_rate": 2})
)

In [ ]:
# график
fig, ax = plt.subplots(figsize=(13, 6))

# Получаем список тарифов из рассчитанной таблицы
tariff_ids = sorted(tariff_churn["tariff_id"].dropna().unique())

# Задаём цвета для тарифов
# Используем словарь, чтобы цвет не зависел от порядка строк
palette_colors = sns.color_palette("Set2", n_colors=len(tariff_ids))
tariff_colors = dict(zip(tariff_ids, palette_colors))

# Строим отдельную линию Churn Rate для каждого тарифа
sns.lineplot(
    data=tariff_churn,
    x="month",
    y="churn_rate",
    hue="tariff_id",
    hue_order=tariff_ids,
    palette=tariff_colors,
    marker="o",
    linewidth=2.5,
    ax=ax
)

# Перебираем тарифы по очереди
for i, tariff_id in enumerate(tariff_ids):

    # Отбираем значения только рассматриваемого тарифа
    tariff_values = tariff_churn[
        tariff_churn["tariff_id"] == tariff_id
    ]

    # Для первого тарифа подписи располагаем выше точек
    # Для второго — ниже, чтобы избежать наложений
    offset = 14 if i % 2 == 0 else -20

    # Добавляем подпись к каждой точке тарифа
    for _, row in tariff_values.iterrows():

        ax.annotate(
            f'{row["churn_rate"]:.2f}%'.replace(".", ","),
            xy=(row["month"], row["churn_rate"]),
            xytext=(0, offset),
            textcoords="offset points",
            ha="center",
            fontsize=9,
            color=tariff_colors[tariff_id]
        )

# Добавляем название и подписи осей
ax.set_title("Коэффициент оттока по тарифам, 2026")
ax.set_xlabel("Месяц")
ax.set_ylabel("Churn Rate, %")

# Подписываем все месяцы
ax.set_xticks(months)
ax.set_xticklabels(month_labels)

# Добавляем легенду тарифов
ax.legend(title="Тариф")

# Увеличиваем вертикальные отступы для подписей
ax.margins(y=0.35)
# Отбираем июльские значения Churn Rate по тарифам
july_tariffs = tariff_churn[
    tariff_churn["month"] == months[-1]
]

# Рассчитываем разницу между максимальным и минимальным Churn Rate
july_difference = (
    july_tariffs["churn_rate"].max()
    - july_tariffs["churn_rate"].min()
)

# Определяем точку с максимальным июльским Churn Rate
july_max_churn = july_tariffs["churn_rate"].max()

# Добавляем стрелку к июльской точке
ax.annotate(
    f"Разница {july_difference:.2f} п.п.".replace(".", ","),
    xy=(months[-1], july_max_churn),
    xytext=(-120, 65),
    textcoords="offset points",
    fontsize=10,
    fontweight="bold",
    arrowprops={
        "arrowstyle": "->",
        "color": "gray",
        "lw": 1.5
    }
)

# Заголовок отражает главный результат
ax.set_title(
    "В июле Churn Rate тарифов различался лишь на 0,03 п.п.",
    fontsize=14,
    fontweight="bold"
)

# Оставляем место для стрелки и подписей
ax.margins(y=0.40)

plt.tight_layout()
plt.show()

Коэффициенты оттока двух тарифов в январе–июле 2026 года находились на близких уровнях. В июле Churn Rate составлял 3,91% для тарифа 1 и 3,94% для тарифа 2. Разница составила около 0,03 процентного пункта.

На графике не наблюдается устойчивого существенного превышения коэффициента оттока одного тарифа над другим.

**Увеличение числа завершений, вероятно, нельзя объяснить выраженным ухудшением удержания в одном конкретном тарифе**. При этом тариф с большей активной базой может давать больше завершений в абсолютном выражении.

**H3, предполагающая существенные различия интенсивности оттока между тарифами, не получает подтверждения** на описательном уровне. H1 остаётся основным направлением дальнейшей проверки.

#### Структура причин завершения подписки
Ответ на вопрос: Какие причины завершения подписок встречаются чаще всего и отличаются ли они между тарифами?

In [ ]:
# Отбираем подписки, завершившиеся в январе–июле 2026 года
ended_2026 = mart[
    # Дата завершения не раньше 1 января
    (mart["ended_at"] >= "2026-01-01")

    # Дата завершения раньше 1 августа
    & (mart["ended_at"] < "2026-08-01")
].copy()

# Проверяем, сколько завершений не имеют указанной причины
missing_reasons = ended_2026["cancel_reason"].isna().sum()

# Выводим результат проверки
print("Завершений без причины:", missing_reasons)

# Заменяем пропущенные причины отдельной категорией
# Это позволяет не потерять такие подписки при группировке
ended_2026["cancel_reason"] = (
    ended_2026["cancel_reason"].fillna("Не указана")
)

# Группируем завершения по тарифу и причине
reason_metrics = (
    ended_2026
    .groupby(
        ["tariff_id", "cancel_reason"],
        as_index=False
    )

    # Считаем количество уникальных подписок в каждой группе
    .agg(
        subscriptions_count=("subscription_id", "nunique")
    )
)

# Рассчитываем долю каждой причины внутри тарифа
reason_metrics["share_pct"] = (
    reason_metrics["subscriptions_count"]

    # Получаем общее число завершений соответствующего тарифа
    / reason_metrics.groupby("tariff_id")[
        "subscriptions_count"
    ].transform("sum")

    # Переводим долю в проценты
    * 100
)

# Выводим количество и долю завершений по каждой причине
display(
    reason_metrics.round({"share_pct": 2})
)

In [ ]:
# график
# Строки — причины завершения
# Столбцы — тарифы
# Значения — доля причины внутри тарифа
reason_pivot = reason_metrics.pivot(
    index="cancel_reason",
    columns="tariff_id",
    values="share_pct"
)

# Заменяем отсутствующие сочетания причины и тарифа на 0
# Это означает, что по данной причине не было завершений
reason_pivot = reason_pivot.fillna(0)

# Создаём отдельную таблицу текстовых подписей
# Например, число 33.76 превращается в строку "33,8%"
reason_labels = reason_pivot.map(
    lambda value: f"{value:.1f}%".replace(".", ",")
)

# Создаём рисунок
fig, ax = plt.subplots(figsize=(10, 6))

# Строим тепловую карту долей
sns.heatmap(
    reason_pivot,

    # Выводим процент в каждой ячейке
    annot=reason_labels,

    # Используем готовые строковые подписи
    fmt="",

    # Задаём цветовую шкалу
    cmap="Blues",

    # Добавляем название шкалы цветов
    cbar_kws={"label": "Доля завершений, %"},

    ax=ax
)

# Добавляем название и подписи осей
ax.set_title("Причины завершения подписок по тарифам, 2026")
ax.set_xlabel("Тариф")
ax.set_ylabel("Причина завершения")

# Корректируем расположение элементов
plt.tight_layout()
plt.show()

Структура зарегистрированных причин завершения подписок в январе–июле 2026 года практически одинакова для обоих тарифов.

Наиболее распространённая причина — `not_using` (около 34% завершений), на втором месте — `too_expensive` (около 24%). Совместно эти причины составляют приблизительно 58% завершений в каждом тарифе.

**Связь с бизнес-вопросом:** анализ показывает, какие причины наиболее распространены среди завершившихся подписок, но не объясняет сам по себе увеличение количества завершений между январём и июлем. Для этого потребовалось бы дополнительно исследовать изменение структуры причин по месяцам.

**Связь с гипотезами:** часть H3, предполагающая выраженные различия структуры причин между тарифами, не подтверждается. При этом высокая доля причин `not_using` и `too_expensive` создаёт основу для рекомендаций по повышению вовлечённости пользователей и воспринимаемой ценности подписки.

In [ ]:
# Проверяем, что чистый прирост равен разности
# новых и завершившихся подписок
assert (
    net_growth["net_growth"]
    == net_growth["new_subscriptions"]
    - net_growth["ended_subscriptions"]
).all(), "Ошибка расчёта чистого прироста"

# Проверяем, что коэффициент оттока находится в пределах 0–100%
assert churn_metrics["churn_rate"].between(0, 100).all(), \
    "Некорректное значение Churn Rate"

# Проверяем, что число завершений из начальной базы
# не превышает общего количества завершений за месяц
assert (
    churn_metrics["churned_from_start"].to_numpy()
    <= subscriptions_dynamics["ended_subscriptions"].to_numpy()
).all(), "Некорректное количество завершений из начальной базы"

# Суммируем завершения из начальной базы по тарифам
tariff_churn_total = (
    tariff_churn
    .groupby("month")["churned_from_start"]
    .sum()
    .sort_index()
)

# Получаем общее количество завершений из начальной базы
overall_churn_total = (
    churn_metrics
    .set_index("month")["churned_from_start"]
    .sort_index()
)

# Проверяем совпадение общих значений с суммой по тарифам
assert tariff_churn_total.equals(overall_churn_total), \
    "Сумма завершений по тарифам не совпадает с общим показателем"

# Проверяем, что суммы долей причин по тарифам равны 100%
reason_share_total = (
    reason_metrics
    .groupby("tariff_id")["share_pct"]
    .sum()
)

assert ((reason_share_total - 100).abs() < 0.000001).all(), \
    "Доли причин завершения не суммируются до 100%"

# Выводим сообщение, если все проверки пройдены
print("Все проверки раздела 2.3 успешно пройдены.")

#### Итоговые результаты анализа метрик

Анализ января–июля 2026 года показал, что количество завершившихся подписок увеличилось с 6 607 до 7 407 (+12,1%), тогда как количество новых подписок практически не изменилось: с 11 326 до 11 338 (+0,1%).

В результате месячный чистый прирост подписочной базы сократился с 4 719 до 3 931 подписки (−16,7%), а соотношение завершений к новым подпискам увеличилось с 58,33% до 65,33%.

При этом активная база на начало месяца выросла примерно на 15%, тогда как коэффициент оттока оставался относительно стабильным: 4,02% в январе и 3,92% в июле. Следовательно, увеличение абсолютного количества завершений не сопровождалось ухудшением общего Churn Rate.

Анализ тарифов не выявил устойчивых существенных различий в интенсивности оттока. Структура зарегистрированных причин завершения также оказалась близкой между тарифами: наиболее распространёнными причинами стали `not_using` и `too_expensive`.

**Результаты проверки гипотез:**

- **H1 — предварительно поддерживается:** рост активной базы при стабильном Churn Rate согласуется с предположением, что увеличение абсолютного числа завершений связано преимущественно с размером базы. Точный вклад факторов ещё предстоит рассчитать.
- **H2 — подтверждается для сравнения января и июля:** увеличение завершений при практически неизменном притоке новых подписок привело к сокращению месячного чистого прироста.
- **H3 — не получила подтверждения на описательном уровне:** существенных устойчивых различий между тарифами по коэффициенту оттока и структуре причин завершения не обнаружено.

Таким образом, основным направлением дальнейшего исследования становится количественная оценка влияния размера активной базы и коэффициента оттока на количество завершений.

В разделе 3 будет выполнена факторная декомпозиция, которая позволит определить, насколько каждый фактор повлиял на изменение числа завершений и чистого прироста подписочной базы.

## Часть 3. Факторный анализ

In [ ]:
# Создаём таблицу для проверки состава завершений
churn_balance = churn_metrics[
    ["month", "active_start", "churned_from_start"]
].copy()

# Добавляем общее количество завершений за месяц
churn_balance["ended_subscriptions"] = (
    subscriptions_dynamics["ended_subscriptions"].to_numpy()
)

# Рассчитываем завершения подписок,
# которые начались и завершились в одном месяце
churn_balance["ended_same_month"] = (
    churn_balance["ended_subscriptions"]
    - churn_balance["churned_from_start"]
)

# Проверяем, что число таких завершений неотрицательно
assert (churn_balance["ended_same_month"] >= 0).all(), \
    "Ошибка: завершений из начальной базы больше, чем всех завершений"

# Проверяем равенство двух частей общему числу завершений
assert (
    churn_balance["ended_subscriptions"]
    == churn_balance["churned_from_start"]
    + churn_balance["ended_same_month"]
).all(), "Ошибка баланса завершений"

# Выводим результат
display(churn_balance)

print("Баланс завершений подтверждён.")

### 3.1. Изменение чистого прироста подписочной базы

В разделе 2.3 было обнаружено, что месячный чистый прирост подписок сократился с 4 719 до 3 931.

Теперь определим, что именно вызвало это сокращение: изменение притока новых подписок или увеличение завершений.

Используем аддитивную модель: **G=A-E**

где G — чистый прирост, A — новые подписки, E — завершившиеся подписки.

Изменение чистого прироста:
**Delta G=(A1-A0)-(E1-E0)**

Это позволяет проверить **H2**: сокращение чистого прироста преимущественно связано с увеличением завершений, а не изменением притока новых подписок.

In [ ]:
# Берём показатели января — первого месяца исследования
jan = net_growth.iloc[0]

# Берём показатели июля — последнего месяца исследования
jul = net_growth.iloc[-1]

# Рассчитываем фактическое изменение чистого прироста
# Вычитаем январское значение из июльского
delta_net_growth = (
    jul["net_growth"] - jan["net_growth"]
)

# Рассчитываем вклад изменения новых подписок
# Положительное значение увеличивает чистый прирост
effect_new = (
    jul["new_subscriptions"] - jan["new_subscriptions"]
)

# Рассчитываем вклад изменения завершившихся подписок
# Ставим минус, поскольку завершения уменьшают чистый прирост
effect_ended = -(
    jul["ended_subscriptions"] - jan["ended_subscriptions"]
)

# Создаём таблицу с результатами факторного анализа
net_factors = pd.DataFrame({
    "factor": [
        "Изменение новых подписок",
        "Изменение завершений"
    ],
    "effect": [
        effect_new,
        effect_ended
    ]
})

# Проверяем, что сумма эффектов равна фактическому изменению
assert net_factors["effect"].sum() == delta_net_growth, \
    "Ошибка: сумма факторов не совпадает с изменением прироста"

# Выводим таблицу факторов
display(net_factors)

# Выводим итоговое изменение чистого прироста
print("Изменение чистого прироста:", delta_net_growth)

Аддитивная декомпозиция показала, что увеличение количества завершений на 800 подписок уменьшило чистый прирост на 800. При этом количество новых подписок выросло на 12, что оказало небольшой положительный эффект.

Сокращение чистого прироста практически объясняется увеличением количества завершившихся подписок, а не снижением притока новых. **Гипотеза Н2 подтверждается** для сравнения января и июля 2026 года. Отрицательный вклад увеличения завершений (−800) существенно превышает положительный вклад изменения новых подписок (+12).

Следующий этап анализа — определить, почему увеличилось количество завершений.

### 3.2. Факторный анализ роста завершившихся подписок

Теперь необходимо выяснить, почему завершений стало больше.

Чтобы ответить, используем модель: **E=N*c+W**

где:
- \(E\) — все завершения за месяц;
- \(N\) — активная база на начало месяца;
- \(c\) — Churn Rate, выраженный долей единицы;
- \(W\) — подписки, начавшиеся и завершившиеся внутри одного месяца.
Компонент \(W\) необходим, поскольку такие подписки учитываются в общем числе завершений, но не входят в начальную активную базу.

Эта модель позволяет проверить **H1**: увеличение завершений преимущественно объясняется расширением активной базы, а не ухудшением коэффициента оттока.

In [ ]:
# Берём показатели января из таблицы баланса завершений
jan_churn = churn_balance.iloc[0]

# Берём показатели июля из той же таблицы
jul_churn = churn_balance.iloc[-1]

# Размер активной базы на начало января
base_0 = jan_churn["active_start"]

# Размер активной базы на начало июля
base_1 = jul_churn["active_start"]

# Рассчитываем январский Churn Rate как долю
# Используем исходные значения, без округления до 4,02%
churn_0 = (
    jan_churn["churned_from_start"] / base_0
)

# Аналогично рассчитываем июльский Churn Rate
churn_1 = (
    jul_churn["churned_from_start"] / base_1
)

# Количество завершений внутри месяца начала в январе
same_month_0 = jan_churn["ended_same_month"]

# Количество завершений внутри месяца начала в июле
same_month_1 = jul_churn["ended_same_month"]

# Фактическое изменение общего количества завершений
delta_ended = (
    jul_churn["ended_subscriptions"]
    - jan_churn["ended_subscriptions"]
)

# Выводим исходные значения для проверки
print("Активная база:", base_0, "→", base_1)
print("Churn Rate:", f"{churn_0:.4%}", "→", f"{churn_1:.4%}")
print("Завершения внутри месяца:", same_month_0, "→", same_month_1)
print("Изменение всех завершений:", delta_ended)

In [ ]:
# ФАКТОР 1: изменение размера активной базы
# Изменение базы умножаем на январский коэффициент оттока
effect_base = (
    (base_1 - base_0) * churn_0
)

# ФАКТОР 2: изменение коэффициента оттока
# Изменение Churn Rate умножаем на июльскую активную базу
effect_churn = (
    base_1 * (churn_1 - churn_0)
)

# ФАКТОР 3: завершения внутри месяца начала
# Сравниваем количество таких завершений в июле и январе
effect_same_month = (
    same_month_1 - same_month_0
)

# Создаём таблицу результатов факторной декомпозиции
ended_factors = pd.DataFrame({
    "factor": [
        "Рост активной базы",
        "Изменение Churn Rate",
        "Завершения внутри месяца"
    ],
    "effect": [
        effect_base,
        effect_churn,
        effect_same_month
    ]
})

# Рассчитываем сумму вкладов всех факторов
total_effect = ended_factors["effect"].sum()

# Проверяем, что сумма совпадает с фактическим изменением
# Используем небольшую погрешность из-за дробных вычислений
assert abs(total_effect - delta_ended) < 0.000001, \
    "Ошибка: факторная декомпозиция не сходится"

# Выводим результаты с округлением до двух знаков
display(ended_factors.round({"effect": 2}))

# Выводим общий результат
print("Сумма факторных эффектов:", round(total_effect, 2))
print("Фактическое изменение завершений:", delta_ended)

Рост абсолютного числа завершений преимущественно связан с расширением активной подписочной базы, а не с увеличением интенсивности оттока. При большей базе даже относительно стабильный Churn Rate приводит к большему количеству завершений.

**Гипотеза Н1 подтверждается** для сравнения января и июля 2026 года. Вклад увеличения активной базы (+990) существенно превышает по абсолютной величине вклад изменения Churn Rate (−191).

Таким образом, основным фактором роста завершений является размер активной базы, а снижение Churn Rate частично сдерживает этот рост.

In [ ]:
# Создаём итоговую таблицу факторов чистого прироста
final_factors = pd.DataFrame({
    "factor": [
        "Изменение новых подписок",
        "Рост активной базы",
        "Изменение Churn Rate",
        "Завершения внутри месяца"
    ],

    # Влияние новых подписок на прирост положительное
    # Влияние завершений на прирост имеет обратный знак
    "effect": [
        effect_new,
        -effect_base,
        -effect_churn,
        -effect_same_month
    ]
})

# Рассчитываем суммарный эффект всех четырёх факторов
final_effect = final_factors["effect"].sum()

# Проверяем совпадение с фактическим изменением чистого прироста
assert abs(final_effect - delta_net_growth) < 0.000001, \
    "Ошибка: итоговая декомпозиция чистого прироста не сходится"

# Выводим итоговую таблицу
display(final_factors.round({"effect": 2}))

# Выводим результат проверки
print("Сумма факторных эффектов:", round(final_effect, 2))
print("Фактическое изменение чистого прироста:", delta_net_growth)

In [ ]:
# Берём январь и июль из таблицы чистого прироста
jan = net_growth.iloc[0]
jul = net_growth.iloc[-1]

# Берём январь и июль из таблицы баланса завершений
jan_churn = churn_balance.iloc[0]
jul_churn = churn_balance.iloc[-1]

# Определяем чистый прирост в январе и июле
growth_0 = jan["net_growth"]
growth_1 = jul["net_growth"]

# Определяем активную базу на начало января и июля
base_0 = jan_churn["active_start"]
base_1 = jul_churn["active_start"]

# Рассчитываем коэффициенты оттока как доли единицы
# Не округляем значения до расчёта факторных эффектов
churn_0 = jan_churn["churned_from_start"] / base_0
churn_1 = jul_churn["churned_from_start"] / base_1

# ФАКТОР 1: изменение притока новых подписок
# Больше новых подписок — выше чистый прирост
effect_new = (
    jul["new_subscriptions"]
    - jan["new_subscriptions"]
)

# ФАКТОР 2: изменение размера активной базы
# Больше база — больше завершений при том же Churn Rate
# Поэтому для чистого прироста ставим отрицательный знак
effect_base = -(
    (base_1 - base_0) * churn_0
)

# ФАКТОР 3: изменение Churn Rate
# Используем июльскую базу после первой подстановки
# Отрицательный знак переводит эффект завершений
# в эффект для чистого прироста
effect_churn = -(
    base_1 * (churn_1 - churn_0)
)

# ФАКТОР 4: завершения подписок внутри месяца начала
# Увеличение таких завершений сокращает чистый прирост
effect_same_month = -(
    jul_churn["ended_same_month"]
    - jan_churn["ended_same_month"]
)

# Создаём итоговую таблицу факторных эффектов
final_factors = pd.DataFrame({
    "factor": [
        "Новые подписки",
        "Рост активной базы",
        "Изменение Churn Rate",
        "Завершения внутри месяца"
    ],
    "effect": [
        effect_new,
        effect_base,
        effect_churn,
        effect_same_month
    ]
})

# Рассчитываем фактическое изменение чистого прироста
actual_change = growth_1 - growth_0

# Проверяем, что сумма факторов равна фактическому изменению
assert abs(final_factors["effect"].sum() - actual_change) < 0.000001, \
    "Ошибка: сумма факторных эффектов не совпадает с изменением прироста"

# Проверяем, что модель завершений совпадает с фактическими данными
assert abs(
    base_0 * churn_0
    + jan_churn["ended_same_month"]
    - jan_churn["ended_subscriptions"]
) < 0.000001, "Ошибка баланса завершений января"

assert abs(
    base_1 * churn_1
    + jul_churn["ended_same_month"]
    - jul_churn["ended_subscriptions"]
) < 0.000001, "Ошибка баланса завершений июля"

# Выводим таблицу с факторными эффектами
display(final_factors.round({"effect": 2}))

# Выводим итог проверки
print("Чистый прирост января:", growth_0)
print("Чистый прирост июля:", growth_1)
print("Изменение:", actual_change)
print("Сумма факторов:", round(final_factors["effect"].sum(), 2))
print("Декомпозиция сошлась.")

In [ ]:
# график
effects = final_factors["effect"].tolist()

# Задаём подписи для начального значения, факторов и итога
labels = [
    "Январь 2026",
    "Новые\nподписки",
    "Активная\nбаза",
    "Churn Rate",
    "Завершения\nвнутри месяца",
    "Июль 2026"
]

# Создаём список накопленных значений
# Первое значение — чистый прирост января
levels = [float(growth_0)]

# Последовательно прибавляем влияние каждого фактора
for effect in effects:
    levels.append(levels[-1] + effect)

# Проверяем, что после всех подстановок получился июль
assert abs(levels[-1] - growth_1) < 0.000001, \
    "Ошибка: каскадный график не сходится"

# Создаём рисунок
fig, ax = plt.subplots(figsize=(14, 6))

# Задаём цвета столбцов
total_color = "#6b7280"       # Серый — начальное и конечное значения
negative_color = "#d64545"    # Красный — отрицательный вклад
positive_color = "#3b9b75"    # Зелёный — положительный вклад

# Задаём ширину столбцов
bar_width = 0.7

# Строим первый столбец — чистый прирост января
ax.bar(
    0,
    growth_0,
    width=bar_width,
    color=total_color
)

# Строим промежуточные столбцы факторных эффектов
for i, effect in enumerate(effects):

    # Уровень до действия фактора
    previous_level = levels[i]

    # Уровень после действия фактора
    next_level = levels[i + 1]

    # Нижняя граница столбца — меньший из двух уровней
    bottom = min(previous_level, next_level)

    # Высота столбца — абсолютная величина эффекта
    height = abs(effect)

    # Выбираем цвет по направлению влияния
    color = positive_color if effect >= 0 else negative_color

    # Рисуем столбец, который начинается не от нуля,
    # а от соответствующего уровня накопленного результата
    ax.bar(
        i + 1,
        height,
        bottom=bottom,
        width=bar_width,
        color=color
    )

    # Добавляем тонкую линию между соседними столбцами
    # Она показывает уровень после предыдущего фактора
    ax.plot(
        [i + bar_width / 2, i + 1 - bar_width / 2],
        [previous_level, previous_level],
        color="gray",
        linewidth=1,
        linestyle="--"
    )

# Строим последний столбец — чистый прирост июля
ax.bar(
    len(effects) + 1,
    growth_1,
    width=bar_width,
    color=total_color
)

# Создаём функцию для форматирования чисел
# Например, 4719 превращается в "4 719"
def format_number(value, decimals=0):
    return (
        f"{value:,.{decimals}f}"
        .replace(",", " ")
        .replace(".", ",")
    )

# Подписываем первый столбец
ax.text(
    0,
    growth_0 + 70,
    format_number(growth_0),
    ha="center",
    fontsize=11,
    fontweight="bold"
)

# Подписываем каждое факторное изменение
for i, effect in enumerate(effects):

    # Определяем границы факторного столбца
    previous_level = levels[i]
    next_level = levels[i + 1]

    # Положительные эффекты подписываем над столбцом
    # Отрицательные — под столбцом
    if effect >= 0:
        y_position = max(previous_level, next_level) + 55
        vertical_align = "bottom"
    else:
        y_position = min(previous_level, next_level) - 55
        vertical_align = "top"

    # Для дробных эффектов показываем два знака после запятой
    # Для целых — без десятичных знаков
    decimals = 0 if abs(effect - round(effect)) < 0.000001 else 2

    # Формируем подпись со знаком плюс или минус
    effect_label = (
        ("+" if effect > 0 else "")
        + format_number(effect, decimals)
    )

    # Добавляем подпись фактора
    ax.text(
        i + 1,
        y_position,
        effect_label,
        ha="center",
        va=vertical_align,
        fontsize=10,
        fontweight="bold"
    )

# Подписываем последний столбец
ax.text(
    len(effects) + 1,
    growth_1 + 70,
    format_number(growth_1),
    ha="center",
    fontsize=11,
    fontweight="bold"
)

# Добавляем стрелку к главному отрицательному фактору
# Это второй фактор: рост активной базы
base_effect_level = min(levels[1], levels[2])

ax.annotate(
    "Главный отрицательный фактор",
    xy=(2, base_effect_level + abs(effects[1]) / 2),
    xytext=(3.0, 5100),
    fontsize=10,
    fontweight="bold",
    color=negative_color,
    arrowprops={
        "arrowstyle": "->",
        "color": negative_color,
        "lw": 1.5
    }
)

# Добавляем аналитический заголовок
ax.set_title(
    "Чистый прирост сократился на 788 подписок — "
    "главный фактор: рост активной базы",
    fontsize=13,
    fontweight="bold",
    pad=20
)

# Подписываем ось Y
ax.set_ylabel("Чистый прирост, подписок")

# Устанавливаем подписи столбцов
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)

# Задаём вертикальные границы с запасом для подписей
ax.set_ylim(0, max(levels) * 1.18)

# Убираем лишние рамки
sns.despine(ax=ax)

# Оставляем горизонтальную сетку для удобства чтения
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)

# Корректируем расположение элементов
plt.tight_layout()

# Отображаем график
plt.show()


С января по июль 2026 года месячный чистый прирост подписочной базы ПРАЙМ сократился на 788 подписок.

Наибольший отрицательный вклад внесло расширение активной базы (−990,14 подписки). Однако снижение коэффициента оттока частично компенсировало этот эффект (+191,14 подписки). Увеличение количества новых подписок добавило 12 подписок к чистому приросту, а изменение завершений внутри месяца начала уменьшило его ещё на одну подписку.

Сокращение чистого прироста объясняется увеличением абсолютного числа завершений, которое, в свою очередь, преимущественно связано с расширением активной базы. При этом интенсивность оттока не ухудшилась, а немного снизилась.

**Проверка гипотез:** H1 и H2 подтверждаются для сравнения января и июля 2026 года. H3 о существенных различиях между тарифами оценивалась отдельно в разделе 2.3 и не получила подтверждения на уровне описательных метрик.

## Часть 4. Вывод и рекомендации

### 4.1. Ответ на бизнес-вопрос

**Бизнес-вопрос:** почему с января по июль 2026 года количество завершившихся подписок увеличилось сильнее количества новых подписок и какие факторы определили сокращение месячного чистого прироста подписочной базы ПРАЙМ?

Анализ показал, что количество завершившихся подписок увеличилось с **6 607 до 7 407 (+12,1%)**, тогда как количество новых подписок практически не изменилось — с **11 326 до 11 338 (+0,1%)**. В результате месячный чистый прирост сократился с **4 719 до 3 931 подписки (−16,7%)**. При этом прирост оставался положительным на протяжении всего исследуемого периода.

Факторная декомпозиция показала, что сокращение чистого прироста на 788 подписок обусловлено увеличением количества завершений (вклад −800 подписок), которое лишь частично компенсировалось увеличением притока новых подписок (+12).

Главным фактором роста завершений стало **расширение активной подписочной базы**, которая увеличилась со 163 809 до 188 410 подписок (+15,0%). Метод цепных подстановок показал, что увеличение базы внесло вклад около **+990 завершений**, тогда как снижение Churn Rate с 4,02% до 3,92% сократило их число примерно на 191. Изменение числа подписок, начавшихся и завершившихся в одном месяце, добавило одно завершение.

Таким образом, **основная причина сокращения месячного чистого прироста — увеличение абсолютного количества завершений на фоне роста активной базы, а не ухудшение интенсивности оттока**. По мере расширения базы даже стабильный Churn Rate приводит к большему числу завершений, для компенсации которых необходим соответствующий приток новых подписок.

### 4.2. Результаты проверки гипотез

- **H1 — подтверждена для сравнения января и июля 2026 года.** Увеличение абсолютного количества завершений преимущественно объясняется ростом активной базы, а не ухудшением Churn Rate.
- **H2 — подтверждена.** Сокращение месячного чистого прироста практически полностью связано с увеличением завершений, тогда как изменение притока новых подписок оказало незначительное влияние.
- **H3 — не получила подтверждения на описательном уровне.** Коэффициенты оттока двух тарифов близки: в июле они составляли 3,91% и 3,94%. Структура причин завершения также практически одинакова для обоих тарифов.

### 4.3. Что остаётся неизвестным

Проведённый факторный анализ объясняет изменение показателей между январём и июлем, но не устанавливает причины решений отдельных клиентов о завершении подписки.

Среди зарегистрированных причин завершения наиболее распространены `not_using` (около 34%) и `too_expensive` (около 24%). Вместе они составляют приблизительно 58% завершений в каждом тарифе. Однако их высокая доля сама по себе не доказывает, что именно эти причины обусловили рост числа завершений за исследуемый период.

Для более точного понимания поведения подписчиков необходимо дополнительно исследовать связь между использованием сервисов, продолжительностью подписки, платежами и вероятностью завершения. Также важно оценить изменение структуры причин завершения по месяцам.

### 4.4. Рекомендации бизнесу

**1. Контролировать не только абсолютное количество завершений, но и Churn Rate относительно активной базы.**

Рост завершений при увеличении базы не обязательно означает ухудшение удержания. Поэтому в регулярной отчётности следует совместно отслеживать активную базу на начало месяца, новые подписки, завершения, Churn Rate и чистый прирост. Это позволит своевременно отличать естественное увеличение числа завершений при расширении базы от реального ухудшения удержания.

**2. Усилить работу с вовлечённостью подписчиков.**

Поскольку `not_using` является наиболее распространённой зарегистрированной причиной завершения, целесообразно проверить, связано ли низкое использование сервисов с повышенной вероятностью отмены. Если связь подтвердится, можно протестировать персональные напоминания о преимуществах подписки и рекомендации релевантных сервисов для малоактивных пользователей.

**3. Проверить воспринимаемую ценность подписки.**

Причина `too_expensive` занимает второе место среди завершений. Рекомендуется изучить, какие преимущества подписки используются реже всего и в каких группах клиентов чаще встречается эта причина. На основе результатов можно протестировать коммуникации, объясняющие ценность тарифа, и оценить их влияние на удержание.

**4. Планировать необходимый приток новых подписок с учётом размера действующей базы.**

При расширении активной базы абсолютное количество завершений может увеличиваться даже при стабильном Churn Rate. Поэтому планирование привлечения следует связывать с прогнозируемым количеством завершений и целевым чистым приростом. Это поможет заранее оценивать, сколько новых подписок необходимо для поддержания желаемых темпов роста.

**5. Проверять эффективность мер удержания с помощью пилотных экспериментов.**

Предлагаемые меры по вовлечённости и повышению воспринимаемой ценности пока являются гипотезами для бизнеса, а не доказанными решениями. Их следует тестировать на отдельных группах подписчиков и сравнивать Churn Rate, долю продлений и чистый прирост с контрольными группами.

**Приоритет для бизнеса — поддерживать баланс между привлечением новых подписчиков и удержанием действующих.** Для этого необходимо учитывать размер активной базы при планировании роста и проверять возможности дальнейшего снижения Churn Rate, прежде всего через повышение вовлечённости и воспринимаемой ценности подписки.